In [24]:
# ============================================================
# DHAKA LOCAL BUS SENTIMENT ANALYSIS WITH UNSLOTH + LoRA
# HOW TO USE:
# 1) Run this cell once -> it will install packages and stop.
# 2) Restart runtime.
# 3) Run the SAME cell again -> it will do the full pipeline.
# ============================================================

import sys, subprocess, pkgutil, os

# -----------------------------
# 0. SAFE INSTALL CHECK
# -----------------------------
REQUIRED = {
    "unsloth": "unsloth",
    "trl": "trl",
    "transformers": "transformers",
    "datasets": "datasets",
    "accelerate": "accelerate",
    "peft": "peft",
    "bitsandbytes": "bitsandbytes",
    "sentencepiece": "sentencepiece",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
}

missing = [pip_name for mod_name, pip_name in REQUIRED.items() if pkgutil.find_loader(mod_name) is None]

if missing:
    print("Installing required packages...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "unsloth",
        "unsloth_zoo",
        "trl<0.15.0",
        "transformers",
        "datasets",
        "accelerate",
        "peft",
        "bitsandbytes",
        "sentencepiece",
        "scikit-learn",
        "pandas"
    ])
    raise SystemExit(
        "Packages installed. Please restart the Colab runtime once, then run this same cell again."
    )

/tmp/ipykernel_8939/147405906.py:27: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  missing = [pip_name for mod_name, pip_name in REQUIRED.items() if pkgutil.find_loader(mod_name) is None]


In [25]:
# -----------------------------
# 1. IMPORT LIBRARIES
# -----------------------------
import re
import json
import random
import shutil
import numpy as np
import pandas as pd
import torch

from google.colab import files
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

In [26]:
# -----------------------------
# 2. REPRODUCIBILITY
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy: 2.0.2
pandas: 2.2.2


In [27]:
# -----------------------------
# 3. UPLOAD CSV MANUALLY
# -----------------------------
print("\nUpload your reduced dataset CSV file now...")
uploaded = files.upload()
csv_file = list(uploaded.keys())[0]
print("Uploaded file:", csv_file)


Upload your reduced dataset CSV file now...


Saving dhaka_bus_sentiment.csv to dhaka_bus_sentiment.csv
Uploaded file: dhaka_bus_sentiment.csv


In [28]:
# -----------------------------
# 4. LOAD AND VALIDATE DATASET
# -----------------------------
df = pd.read_csv(csv_file)

print("\nOriginal dataset shape:", df.shape)
print("Columns found:", df.columns.tolist())

required_cols = [
    "response_id",
    "comment_text",
    "age",
    "profession",
    "time_period",
    "crowding_level",
    "waiting_time_min",
    "sentiment_label",
    "primary_category",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].copy()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()

df = df.dropna(subset=["comment_text", "sentiment_label", "primary_category"]).reset_index(drop=True)
df = df[df["comment_text"] != ""].reset_index(drop=True)

df["sentiment_label"] = df["sentiment_label"].astype(str).str.title().str.strip()

category_cleanup = {
    "Staff behaviour": "Staff Behavior",
    "Staff behaviour ": "Staff Behavior",
    "staff behavior": "Staff Behavior",
    "staff behaviour": "Staff Behavior",
}
df["primary_category"] = df["primary_category"].replace(category_cleanup).astype(str).str.strip()

valid_sentiments = {"Positive", "Neutral", "Negative"}
df = df[df["sentiment_label"].isin(valid_sentiments)].reset_index(drop=True)

print("\nCleaned dataset shape:", df.shape)
display(df.head())

print("\nSentiment distribution:")
display(df["sentiment_label"].value_counts())

print("\nPrimary category distribution:")
display(df["primary_category"].value_counts())


Original dataset shape: (1500, 9)
Columns found: ['response_id', 'comment_text', 'age', 'profession', 'time_period', 'crowding_level', 'waiting_time_min', 'sentiment_label', 'primary_category']

Cleaned dataset shape: (1500, 9)


,response_id,comment_text,age,profession,time_period,crowding_level,waiting_time_min,sentiment_label,primary_category
0,1,"I expected a rough commute, but it was actuall...",52,Teacher,Evening Peak,Severe,6,Positive,Comfort
1,2,Accessibility er dike khub kom attention ache....,44,Job Seeker,Morning Peak,Low,35,Negative,Accessibility
2,3,Getting on the bus was difficult because it ba...,55,Factory Worker,Morning Peak,Moderate,51,Negative,Accessibility
3,4,The fare felt fair enough for the distance I t...,21,Shopkeeper,Morning Peak,Severe,11,Positive,Cost
4,5,A calm and respectful attitude from the staff ...,58,Freelancer,Evening Peak,High,5,Positive,Staff Behavior



Sentiment distribution:


,count
sentiment_label,
Negative,789
Neutral,379
Positive,332



Primary category distribution:


,count
primary_category,
Safety,168
Information,165
Availability,157
Staff Behavior,154
Cleanliness,154
Cost,147
Delay,145
Crowding,143
Accessibility,138


In [29]:
# -----------------------------
# 5. HELPER FUNCTIONS
# -----------------------------
VALID_SENTIMENTS = ["Positive", "Neutral", "Negative"]
VALID_CATEGORIES = [
    "Delay", "Safety", "Cleanliness", "Cost", "Accessibility",
    "Availability", "Crowding", "Staff Behavior", "Comfort", "Information"
]

def build_user_prompt(row):
    return f"""Analyze the following Dhaka local bus commuter feedback.

Comment: {row['comment_text']}
Age: {row['age']}
Profession: {row['profession']}
Time period: {row['time_period']}
Crowding level: {row['crowding_level']}
Waiting time (minutes): {row['waiting_time_min']}

Return the result exactly in this format:
Sentiment: <Positive/Neutral/Negative>
Category: <Delay/Safety/Cleanliness/Cost/Accessibility/Availability/Crowding/Staff Behavior/Comfort/Information>"""

def build_assistant_response(row):
    return f"Sentiment: {row['sentiment_label']}\nCategory: {row['primary_category']}"

def row_to_chat_record(row):
    return {
        "messages": [
            {
                "role": "system",
                "content": "You are a transport sentiment analysis assistant specialized in Dhaka local bus commuter feedback."
            },
            {
                "role": "user",
                "content": build_user_prompt(row)
            },
            {
                "role": "assistant",
                "content": build_assistant_response(row)
            }
        ]
    }

def save_jsonl_from_df(dataframe, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in dataframe.iterrows():
            f.write(json.dumps(row_to_chat_record(row), ensure_ascii=False) + "\n")

def normalize_sentiment(x):
    if x is None:
        return "UNKNOWN"
    x = str(x).strip().title()
    return x if x in VALID_SENTIMENTS else "UNKNOWN"

def normalize_category(x):
    if x is None:
        return "UNKNOWN"
    x = str(x).strip().replace("Behaviour", "Behavior")
    x = " ".join([w.capitalize() for w in x.split()])
    return x if x in VALID_CATEGORIES else "UNKNOWN"

def extract_assistant_only_output(decoded_text, prompt_text):
    if decoded_text.startswith(prompt_text):
        return decoded_text[len(prompt_text):].strip()
    return decoded_text.strip()

def clean_prediction_text(text):
    text = text.replace("<|eot_id|>", " ").replace("</s>", " ").strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text

def extract_sentiment_and_category(pred_text):
    pred_text = clean_prediction_text(pred_text)

    sentiment_match = re.search(r"Sentiment\s*:\s*([A-Za-z]+)", pred_text, re.IGNORECASE)
    category_match = re.search(r"Category\s*:\s*([A-Za-z ]+)", pred_text, re.IGNORECASE)

    pred_sent = sentiment_match.group(1).strip().title() if sentiment_match else None
    pred_cat = category_match.group(1).strip().title() if category_match else None

    if pred_cat is not None:
        pred_cat = pred_cat.replace("Behaviour", "Behavior")
        pred_cat = " ".join([w.capitalize() for w in pred_cat.split()])

    if pred_sent not in VALID_SENTIMENTS:
        lower_text = pred_text.lower()
        if "negative" in lower_text:
            pred_sent = "Negative"
        elif "neutral" in lower_text:
            pred_sent = "Neutral"
        elif "positive" in lower_text:
            pred_sent = "Positive"
        else:
            pred_sent = "UNKNOWN"

    if pred_cat not in VALID_CATEGORIES:
        lower_text = pred_text.lower()
        category_found = None
        for cat in VALID_CATEGORIES:
            if cat.lower() in lower_text:
                category_found = cat
                break
        pred_cat = category_found if category_found is not None else "UNKNOWN"

    return pred_sent, pred_cat, pred_text

def build_inference_messages(row):
    return [
        {
            "role": "system",
            "content": "You are a transport sentiment analysis assistant specialized in Dhaka local bus commuter feedback."
        },
        {
            "role": "user",
            "content": build_user_prompt(row)
        }
    ]

def evaluate_model(model, tokenizer, eval_df, max_seq_length=768, sample_limit=None):
    FastLanguageModel.for_inference(model)

    if sample_limit is not None:
        eval_df = eval_df.iloc[:sample_limit].copy().reset_index(drop=True)
    else:
        eval_df = eval_df.copy().reset_index(drop=True)

    rows = []

    for _, row in eval_df.iterrows():
        messages = build_inference_messages(row)

        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            [input_text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_seq_length
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            temperature=0.0,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

        decoded_full = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        generated_text = extract_assistant_only_output(decoded_full, input_text)
        pred_sent, pred_cat, cleaned_output = extract_sentiment_and_category(generated_text)

        true_sent = normalize_sentiment(row["sentiment_label"])
        true_cat = normalize_category(row["primary_category"])

        rows.append({
            "response_id": row["response_id"],
            "comment_text": row["comment_text"],
            "true_sentiment": true_sent,
            "pred_sentiment": pred_sent,
            "true_category": true_cat,
            "pred_category": pred_cat,
            "raw_generated_output": generated_text,
            "cleaned_generated_output": cleaned_output
        })

    pred_df = pd.DataFrame(rows)

    sent_eval_df = pred_df[
        pred_df["pred_sentiment"].isin(VALID_SENTIMENTS) &
        pred_df["true_sentiment"].isin(VALID_SENTIMENTS)
    ].copy()

    cat_eval_df = pred_df[
        pred_df["pred_category"].isin(VALID_CATEGORIES) &
        pred_df["true_category"].isin(VALID_CATEGORIES)
    ].copy()

    if len(sent_eval_df) > 0:
        sentiment_accuracy = accuracy_score(sent_eval_df["true_sentiment"], sent_eval_df["pred_sentiment"])
        sentiment_precision, sentiment_recall, sentiment_f1, _ = precision_recall_fscore_support(
            sent_eval_df["true_sentiment"],
            sent_eval_df["pred_sentiment"],
            average="weighted",
            zero_division=0
        )
    else:
        sentiment_accuracy = sentiment_precision = sentiment_recall = sentiment_f1 = 0.0

    if len(cat_eval_df) > 0:
        category_accuracy = accuracy_score(cat_eval_df["true_category"], cat_eval_df["pred_category"])
        category_precision, category_recall, category_f1, _ = precision_recall_fscore_support(
            cat_eval_df["true_category"],
            cat_eval_df["pred_category"],
            average="weighted",
            zero_division=0
        )
    else:
        category_accuracy = category_precision = category_recall = category_f1 = 0.0

    metrics_df = pd.DataFrame({
        "Task": ["Sentiment Classification", "Issue Category Classification"],
        "Accuracy": [sentiment_accuracy, category_accuracy],
        "Precision_weighted": [sentiment_precision, category_precision],
        "Recall_weighted": [sentiment_recall, category_recall],
        "F1_weighted": [sentiment_f1, category_f1],
        "Valid_rows_used": [len(sent_eval_df), len(cat_eval_df)]
    })

    return pred_df, metrics_df


In [30]:
# -----------------------------
# 6. SAVE FULL JSONL
# -----------------------------
full_jsonl_path = "dhaka_bus_reduced_full.jsonl"
save_jsonl_from_df(df, full_jsonl_path)
print("\nSaved full JSONL:", full_jsonl_path)

# -----------------------------
# 7. 70 / 15 / 15 SPLIT
# -----------------------------
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["sentiment_label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["sentiment_label"]
)

print("\nTrain shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)


Saved full JSONL: dhaka_bus_reduced_full.jsonl

Train shape: (1050, 9)
Validation shape: (225, 9)
Test shape: (225, 9)


In [31]:
# -----------------------------
# 8. CONVERT SPLITS TO HF DATASETS
# -----------------------------
train_records = [row_to_chat_record(row) for _, row in train_df.iterrows()]
valid_records = [row_to_chat_record(row) for _, row in valid_df.iterrows()]
test_records  = [row_to_chat_record(row) for _, row in test_df.iterrows()]

train_dataset = Dataset.from_list(train_records)
valid_dataset = Dataset.from_list(valid_records)

# -----------------------------
# 9. LOAD BASE MODEL
# -----------------------------
max_seq_length = 768
dtype = None
load_in_4bit = True
model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

print("\nLoading base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully.")


Loading base model...
==((====))==  Unsloth 2025.10.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded successfully.


In [32]:
# -----------------------------
# 10. BASE MODEL EVALUATION
# -----------------------------
print("\nEvaluating base model...")
base_pred_df, base_metrics_df = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    eval_df=test_df,
    max_seq_length=max_seq_length,
    sample_limit=min(50, len(test_df))
)

print("\nBase model metrics:")
display(base_metrics_df)

print("\nSample base predictions:")
display(base_pred_df.head())


Evaluating base model...

Base model metrics:


,Task,Accuracy,Precision_weighted,Recall_weighted,F1_weighted,Valid_rows_used
0,Sentiment Classification,0.66,0.552206,0.66,0.586072,50
1,Issue Category Classification,0.14,0.019600,0.14,0.034386,50



Sample base predictions:


,response_id,comment_text,true_sentiment,pred_sentiment,true_category,pred_category,raw_generated_output,cleaned_generated_output
0,342,There seemed to be more buses on the road than...,Positive,Negative,Availability,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
1,474,The fare is not the only issue—the unclear way...,Negative,Negative,Cost,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
2,559,"I found a bus fairly quickly, even during the ...",Positive,Neutral,Availability,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
3,111,"The fare felt acceptable, though service quali...",Neutral,Neutral,Cost,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
4,792,"Compared with other days, today the cost felt ...",Positive,Neutral,Cost,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...


In [33]:
# -----------------------------
# 11. ADD LoRA ADAPTERS
# -----------------------------
print("\nAdding LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=max_seq_length,
    use_rslora=False,
    loftq_config=None,
)
print("LoRA adapters added.")


Adding LoRA adapters...
LoRA adapters added.


In [34]:
# -----------------------------
# 12. FINE-TUNE
# -----------------------------
print("\nStarting fine-tuning...")

def formatting_function(example):
    # example is a single dictionary from the dataset, e.g., {'messages': [...]}
    # Apply the tokenizer's chat template to convert the list of messages into a single string
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    # dataset_text_field="messages", # Removed as formatting_func will handle this
    formatting_func=formatting_function,
    args=SFTConfig(
        output_dir="unsloth_outputs",
        max_seq_length=max_seq_length,
        packing=False,
        num_train_epochs=2,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        learning_rate=1e-4,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        seed=SEED,
    ),
)

train_result = trainer.train()
print("\nTraining complete.")
print(train_result)


Starting fine-tuning...


Map (num_proc=6):   0%|          | 0/1050 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/225 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,050 | Num Epochs = 2 | Total steps = 264
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss,Validation Loss
50,0.543700,0.560800
100,0.445300,0.416650
150,0.237700,0.239511
200,0.146600,0.150383
250,0.131500,0.131830



Training complete.
TrainOutput(global_step=264, training_loss=0.4967055923559449, metrics={'train_runtime': 347.7875, 'train_samples_per_second': 6.038, 'train_steps_per_second': 0.759, 'total_flos': 2094779148140544.0, 'train_loss': 0.4967055923559449, 'epoch': 2.0})


In [35]:
# -----------------------------
# 13. SAVE ADAPTER
# -----------------------------
adapter_dir = "dhaka_bus_lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("\nSaved adapter to:", adapter_dir)


Saved adapter to: dhaka_bus_lora_adapter


In [36]:
# -----------------------------
# 14. FINE-TUNED MODEL EVALUATION
# -----------------------------
print("\nEvaluating fine-tuned model...")
ft_pred_df, ft_metrics_df = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    eval_df=test_df,
    max_seq_length=max_seq_length,
    sample_limit=min(50, len(test_df))
)

print("\nFine-tuned model metrics:")
display(ft_metrics_df)

print("\nSample fine-tuned predictions:")
display(ft_pred_df.head())

# -----------------------------
# 15. COMPARE BASE VS FINE-TUNED
# -----------------------------
base_metrics_df["Model"] = "Base Model"
ft_metrics_df["Model"] = "Fine-Tuned Model"

comparison_df = pd.concat([base_metrics_df, ft_metrics_df], ignore_index=True)
comparison_df = comparison_df[
    ["Model", "Task", "Accuracy", "Precision_weighted", "Recall_weighted", "F1_weighted", "Valid_rows_used"]
]

print("\nBase vs Fine-Tuned comparison:")
display(comparison_df)

# -----------------------------
# 16. SAVE OUTPUT FILES
# -----------------------------
base_pred_df.to_csv("base_model_predictions.csv", index=False)
ft_pred_df.to_csv("finetuned_model_predictions.csv", index=False)
base_metrics_df.to_csv("base_model_metrics.csv", index=False)
ft_metrics_df.to_csv("finetuned_model_metrics.csv", index=False)
comparison_df.to_csv("base_vs_finetuned_comparison.csv", index=False)

save_jsonl_from_df(train_df, "train.jsonl")
save_jsonl_from_df(valid_df, "valid.jsonl")
save_jsonl_from_df(test_df, "test.jsonl")

shutil.make_archive("dhaka_bus_lora_adapter", "zip", adapter_dir)

print("\nSaved files:")
print("- dhaka_bus_reduced_full.jsonl")
print("- train.jsonl")
print("- valid.jsonl")
print("- test.jsonl")
print("- base_model_predictions.csv")
print("- finetuned_model_predictions.csv")
print("- base_model_metrics.csv")
print("- finetuned_model_metrics.csv")
print("- base_vs_finetuned_comparison.csv")
print("- dhaka_bus_lora_adapter.zip")

# -----------------------------
# 17. DOWNLOAD FILES
# -----------------------------
files.download("dhaka_bus_reduced_full.jsonl")
files.download("train.jsonl")
files.download("valid.jsonl")
files.download("test.jsonl")
files.download("base_model_predictions.csv")
files.download("finetuned_model_predictions.csv")
files.download("base_model_metrics.csv")
files.download("finetuned_model_metrics.csv")
files.download("base_vs_finetuned_comparison.csv")
files.download("dhaka_bus_lora_adapter.zip")


Evaluating fine-tuned model...

Fine-tuned model metrics:


,Task,Accuracy,Precision_weighted,Recall_weighted,F1_weighted,Valid_rows_used
0,Sentiment Classification,1.00,1.0000,1.00,1.000000,50
1,Issue Category Classification,0.14,0.0196,0.14,0.034386,50



Sample fine-tuned predictions:


,response_id,comment_text,true_sentiment,pred_sentiment,true_category,pred_category,raw_generated_output,cleaned_generated_output
0,342,There seemed to be more buses on the road than...,Positive,Positive,Availability,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
1,474,The fare is not the only issue—the unclear way...,Negative,Negative,Cost,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
2,559,"I found a bus fairly quickly, even during the ...",Positive,Positive,Availability,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
3,111,"The fare felt acceptable, though service quali...",Neutral,Neutral,Cost,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
4,792,"Compared with other days, today the cost felt ...",Positive,Positive,Cost,Delay,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...



Base vs Fine-Tuned comparison:


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>